In [1]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.16.8-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = r"C:\Users\shivn\DE-Interview-Prep\.venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\shivn\DE-Interview-Prep\.venv\Scripts\python.exe"

if "SPARK_HOME" in os.environ:
    del os.environ["SPARK_HOME"]

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BTS Flight Delay - Q1 2024") \
    .master("local[*]") \
    .getOrCreate()

print("Spark is running!")
print(f"Spark version: {spark.version}")

Spark is running!
Spark version: 3.5.1


In [2]:
jan = spark.read.csv("../data/raw/January_2024.csv", header=True, inferSchema=True)
feb = spark.read.csv("../data/raw/February_2024.csv", header=True, inferSchema=True)
mar = spark.read.csv("../data/raw/March_2024.csv", header=True, inferSchema=True)

df = jan.union(feb).union(mar)

print(f"January rows:  {jan.count():,}")
print(f"February rows: {feb.count():,}")
print(f"March rows:    {mar.count():,}")
print(f"Total Q1 2024: {df.count():,}")
print(f"Total columns: {len(df.columns)}")

January rows:  547,271
February rows: 519,221
March rows:    591,767
Total Q1 2024: 1,658,259
Total columns: 37


In [3]:
df.show(5)

+----+-----+------------+-----------+--------------------+-----------------+--------+-----------------+------+----------------+----------------+----+--------------+--------------+------------+--------+---------+-------------+---------+------------+--------+---------+-------------+---------+---------+-----------------+--------+----------------+-------------------+--------+-------+--------+-------------+-------------+---------+--------------+-------------------+
|YEAR|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|             FL_DATE|OP_UNIQUE_CARRIER|TAIL_NUM|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|ORIGIN_STATE_ABR|DEST|DEST_CITY_NAME|DEST_STATE_ABR|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|DEP_DELAY_NEW|DEP_DEL15|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|ARR_DELAY_NEW|ARR_DEL15|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|FLIGHTS|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|
+----+-----+------------+-----------+--------------------+------------

In [5]:
# Quick data health check
print(f"Total rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nNull counts per column:")
from pyspark.sql.functions import col, sum as spark_sum
df.select([spark_sum(col(c).isNull().cast("int")).alias(c) 
           for c in df.columns]).show(vertical=True)

Total rows: 1,658,259
Total columns: 37

Null counts per column:
-RECORD 0----------------------
 YEAR                | 0       
 MONTH               | 0       
 DAY_OF_MONTH        | 0       
 DAY_OF_WEEK         | 0       
 FL_DATE             | 0       
 OP_UNIQUE_CARRIER   | 0       
 TAIL_NUM            | 6570    
 OP_CARRIER_FL_NUM   | 0       
 ORIGIN              | 0       
 ORIGIN_CITY_NAME    | 0       
 ORIGIN_STATE_ABR    | 0       
 DEST                | 0       
 DEST_CITY_NAME      | 0       
 DEST_STATE_ABR      | 0       
 CRS_DEP_TIME        | 0       
 DEP_TIME            | 27557   
 DEP_DELAY           | 27679   
 DEP_DELAY_NEW       | 27679   
 DEP_DEL15           | 27679   
 CRS_ARR_TIME        | 0       
 ARR_TIME            | 29028   
 ARR_DELAY           | 32207   
 ARR_DELAY_NEW       | 32207   
 ARR_DEL15           | 32207   
 CANCELLED           | 0       
 CANCELLATION_CODE   | 1629748 
 DIVERTED            | 0       
 CRS_ELAPSED_TIME    | 1       
 ACTUAL

In [8]:
# Verify the %  NULL pattern
total_rows = 1_658_259
delay_cause_nulls = 1_327_781
null_percentage = delay_cause_nulls / total_rows * 100
print(f"Delay cause NULL %: {null_percentage:.1f}%")

Delay cause NULL %: 80.1%
